In [1]:
from importlib.metadata import version

from debugpy.launcher import output

print(version("torch"))

2.8.0+cu129


RMSNorm的手写实现

In [2]:
import torch
import torch.nn as nn

In [6]:
class RMSNorm(nn.Module):
    '''RMSNorm模块的核心实现'''
    def __init__(self,dim):
        super().__init__()
        '''RMSNorm模块的核心实现
          x/rms(x) * gamma
          '''
        #可学习的缩放参数
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self,x):
        # x的shape = [batch_size,seq_len,dim]
        rms = torch.sqrt(torch.mean(x**2,dim=-1,keepdim=True))

        #归一化操作
        return x/rms*self.scale



Case1

In [7]:
#测试RMSNorm
batch_size,seq_len,dim = 2,8,128

x=torch.randn(batch_size,seq_len,dim)
print(x)
print(x.shape)

tensor([[[-1.3533,  0.3352, -1.4495,  ...,  0.0994,  0.1641,  0.1643],
         [-1.3780,  0.1035,  1.6531,  ...,  0.0354, -0.2299,  0.1904],
         [-0.0712, -0.7566,  1.7770,  ...,  0.1965, -0.1889, -0.0690],
         ...,
         [-0.2286, -0.9320, -0.3101,  ..., -0.7645,  1.3728, -0.2043],
         [-1.2877,  0.6711, -0.2048,  ..., -1.2776,  0.9260,  0.5274],
         [-0.7280, -0.0988, -0.9638,  ..., -0.6284, -0.8536,  0.7262]],

        [[-0.9264, -1.6371, -1.2809,  ...,  1.6641,  1.1874, -0.4504],
         [-0.0706, -1.2026,  0.7559,  ...,  0.1226, -0.4758,  0.9308],
         [ 0.3781,  0.3299,  0.1731,  ..., -1.7281, -1.6497, -0.9626],
         ...,
         [-0.9510, -0.0895,  1.9550,  ...,  0.8627,  0.3233,  0.1595],
         [ 0.1836, -1.4783, -0.2490,  ..., -0.2396,  0.2719,  1.5894],
         [ 0.9593, -0.6276, -0.6609,  ..., -1.0505, -0.0291, -1.3053]]])
torch.Size([2, 8, 128])


In [8]:
rmsnorm = RMSNorm(dim)
output = rmsnorm(x)
print(output)
print(output.shape)

tensor([[[-1.4735,  0.3649, -1.5782,  ...,  0.1082,  0.1786,  0.1789],
         [-1.3522,  0.1016,  1.6222,  ...,  0.0347, -0.2256,  0.1868],
         [-0.0711, -0.7563,  1.7761,  ...,  0.1964, -0.1888, -0.0690],
         ...,
         [-0.2559, -1.0430, -0.3470,  ..., -0.8556,  1.5364, -0.2286],
         [-1.2780,  0.6660, -0.2033,  ..., -1.2679,  0.9189,  0.5234],
         [-0.7490, -0.1016, -0.9916,  ..., -0.6466, -0.8783,  0.7471]],

        [[-0.9053, -1.5997, -1.2516,  ...,  1.6261,  1.1602, -0.4401],
         [-0.0750, -1.2775,  0.8030,  ...,  0.1302, -0.5054,  0.9888],
         [ 0.3723,  0.3249,  0.1705,  ..., -1.7018, -1.6246, -0.9479],
         ...,
         [-1.0183, -0.0959,  2.0933,  ...,  0.9237,  0.3462,  0.1708],
         [ 0.1788, -1.4394, -0.2424,  ..., -0.2333,  0.2647,  1.5476],
         [ 0.9030, -0.5908, -0.6221,  ..., -0.9889, -0.0274, -1.2287]]],
       grad_fn=<MulBackward0>)
torch.Size([2, 8, 128])


In [9]:
print("output mean: ", output.mean(dim=-1))
print("output std: ", output.std(dim=-1))

output mean:  tensor([[ 0.0771,  0.1021, -0.0297,  0.0709, -0.0033,  0.1407, -0.1743, -0.0821],
        [-0.0032,  0.1914, -0.1420, -0.1017,  0.0152,  0.0196,  0.0437, -0.0636]],
       grad_fn=<MeanBackward1>)
output std:  tensor([[1.0009, 0.9987, 1.0035, 1.0014, 1.0039, 0.9939, 0.9886, 1.0005],
        [1.0039, 0.9854, 0.9938, 0.9987, 1.0038, 1.0037, 1.0030, 1.0019]],
       grad_fn=<StdBackward0>)
